# 06 - 本地 RQAlphaPlus 策略回测
本 Notebook **只在本地授权环境运行**，不在 Colab 运行。先下载 Colab 总 Notebook 生成的 `alphamining_colab_outputs.zip`，放到仓库根目录；本 Notebook 会解压并校验 LightGBM 融合分数。本项目不使用自研回测器。

In [ ]:
from pathlib import Path
import os
if not Path('configs/training_config.yaml').exists():
    repo=os.environ.get('ALPHAMINING_REPO_URL','https://github.com/JacksonChiy/AlphaMining-GFlowNet-AlphaEval.git')
    !git clone $repo
    %cd AlphaMining-GFlowNet-AlphaEval

In [ ]:
%pip install -q -r requirements.txt
# rqalpha_plus 需要通过米筐授权渠道在本地 Python 环境中独立安装。

## 导入 Colab 训练产物
将 `alphamining_colab_outputs.zip` 放在仓库根目录。若已经手工解压且预测文件存在，本单元格不会重复解压。RQAlphaPlus 策略实际读取因子经过 LightGBM 融合后的 `prediction_score.csv`。

In [ ]:
from zipfile import ZipFile

archive_path=Path('alphamining_colab_outputs.zip')
prediction_path=Path('results/lightgbm/prediction_score.csv')
if not prediction_path.exists():
    if not archive_path.exists():
        raise FileNotFoundError('请把 Colab 下载的 alphamining_colab_outputs.zip 放到仓库根目录')
    with ZipFile(archive_path) as archive:
        archive.extractall('.')
assert prediction_path.exists(), '压缩包中缺少 prediction_score.csv'
print('已加载 Colab 预测分数:', prediction_path.resolve())

In [ ]:
import yaml
config=yaml.safe_load(open('configs/training_config.yaml',encoding='utf-8'))
config['backtest']

In [ ]:
RQALPHA_PLUS_BUNDLE='~/.rqalpha-plus/bundle'  # 修改为本地已授权的数据包路径
!python -m rqalpha_strategy.run_backtest --bundle $RQALPHA_PLUS_BUNDLE --predictions results/lightgbm/prediction_score.csv --output-dir results/backtest_report

In [ ]:
from IPython.display import Image,display
display(Image('results/backtest_report/equity_curve.png'))